# Subtractive Genomics Analysis of *Klebsiella pneumoniae*

This notebook documents the **downstream Google Colab analysis** of the *K. pneumoniae* subtractive-genomics workflow.

### Upstream preprocessing
- Reference proteome: **NCBI RefSeq GCF_000240185.1**, *K. pneumoniae* subsp. *pneumoniae* HS11286.
- Initial protein sequences: **5,779**
- CD-HIT redundancy reduction: **5,637 clusters**
- CD-HIT preprocessing was performed in WSL before this notebook.
- The resulting non-redundant proteome, `kp_nr.fasta`, is the starting input for this notebook.

The workflow below performs host-homology filtering, Swiss-Prot homology filtering, candidate annotation/prioritization, and extraction of the selected target sequences.


## 1. Input and software setup

Upload the previously generated non-redundant *K. pneumoniae* protein FASTA (`kp_nr.fasta`).


In [ ]:
from google.colab import files
uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print(filename)


In [ ]:
# Install BLAST+
!apt-get update -qq
!apt-get install -y -qq ncbi-blast+


## 2. Human-protein homology filtering

A reviewed human protein FASTA was retrieved from UniProt and converted into a local BLAST protein database.


In [ ]:
# Download the human UniProt proteome (taxon 9606)
!wget -q "https://rest.uniprot.org/uniprotkb/stream?compressed=true&format=fasta&query=organism_id:9606" -O human.fasta.gz
!gunzip -f human.fasta.gz

!head -3 human.fasta


In [ ]:
# Build the BLAST protein database
!makeblastdb -in human.fasta -dbtype prot -out human_db


### BLASTp: *K. pneumoniae* versus human proteome


In [ ]:
!blastp -query kp_nr.fasta -db human_db -out kp_vs_human.txt -evalue 1e-3 -outfmt 6 -num_threads 2


In [ ]:
# Basic output check
!ls -lh kp_nr.fasta kp_vs_human.txt


### Identify proteins with human homologs

Query IDs appearing in the BLAST output are treated as proteins with detectable human-protein matches and are removed from the *K. pneumoniae* proteome.


In [ ]:
matched_query_ids = set()

with open("kp_vs_human.txt", "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts:
            matched_query_ids.add(parts[0])

print(f"Unique K. pneumoniae proteins with human-protein matches: {len(matched_query_ids)}")


In [ ]:
non_human_sequences_data = []
current_header = ""
current_sequence = []

with open("kp_nr.fasta", "r") as infile:
    for line in infile:
        line = line.strip()

        if line.startswith(">"):
            if current_header and current_sequence:
                sequence_id = current_header[1:].split()[0]
                if sequence_id not in matched_query_ids:
                    non_human_sequences_data.extend(
                        [current_header, "".join(current_sequence)]
                    )

            current_header = line
            current_sequence = []
        else:
            current_sequence.append(line)

    # Process the final sequence
    if current_header and current_sequence:
        sequence_id = current_header[1:].split()[0]
        if sequence_id not in matched_query_ids:
            non_human_sequences_data.extend(
                [current_header, "".join(current_sequence)]
            )

output_filename = "kp_non_human.fasta"
with open(output_filename, "w") as outfile:
    for item in non_human_sequences_data:
        outfile.write(item + "\n")

print(f"Non-human protein FASTA written to: {output_filename}")


## 3. Save intermediate human-filtering results

Results are copied to Google Drive so that the workflow can be resumed across Colab sessions.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil

output_folder = "/content/drive/MyDrive/Subtractive_Genomics_KP_Results"
os.makedirs(output_folder, exist_ok=True)

for filename in ["kp_nr.fasta", "kp_vs_human.txt", "kp_non_human.fasta"]:
    if os.path.exists(filename):
        shutil.copy(filename, os.path.join(output_folder, filename))
        print(f"Saved: {filename}")


## 4. Swiss-Prot homology filtering

The non-human protein set is compared against the reviewed UniProt/Swiss-Prot protein collection. The purpose is to identify proteins lacking significant matches in a highly curated reference protein set.

Swiss-Prot was used here in place of DEG (the Database of Essential Genes), which was the original plan but was dropped due to download access issues. This step should therefore **not** be read as an essentiality filter — it removes proteins with strong matches in a curated, well-characterized reference set, a different (and weaker) criterion than predicted essentiality.


In [ ]:
# Download reviewed UniProt/Swiss-Prot proteins
!wget -q "https://rest.uniprot.org/uniprotkb/stream?query=reviewed:true&format=fasta" -O swissprot.fasta

!head -3 swissprot.fasta


In [ ]:
# Build the Swiss-Prot BLAST database
!makeblastdb -in swissprot.fasta -dbtype prot -out swiss_db


In [ ]:
# Retrieve the non-human FASTA from Drive if it is not already in the session
if not os.path.exists("kp_non_human.fasta"):
    shutil.copy(
        os.path.join(output_folder, "kp_non_human.fasta"),
        "kp_non_human.fasta"
    )


In [ ]:
!blastp -query kp_non_human.fasta -db swiss_db -out kp_vs_swiss.txt -evalue 1e-5 -outfmt 6 -num_threads 2 -max_target_seqs 5


In [ ]:
!ls -lh kp_vs_swiss.txt


## 5. Remove proteins with Swiss-Prot matches

The first BLAST column contains the *K. pneumoniae* query ID. These matched query IDs are removed from `kp_non_human.fasta` to generate the final non-homologous protein set.


In [ ]:
# Install SeqKit
!wget -q https://github.com/shenwei356/seqkit/releases/download/v2.5.0/seqkit_linux_amd64.tar.gz
!tar -xzf seqkit_linux_amd64.tar.gz
!mv -f seqkit /usr/local/bin/seqkit
!chmod +x /usr/local/bin/seqkit

!seqkit version


In [ ]:
!cut -f1 kp_vs_swiss.txt | sort | uniq > matched_ids.txt

!seqkit grep -v -f matched_ids.txt kp_non_human.fasta > kp_final.fasta


In [ ]:
print("Non-human proteins:")
!grep -c ">" kp_non_human.fasta

print("Final non-homologous proteins:")
!grep -c ">" kp_final.fasta


In [ ]:
for filename in ["kp_vs_swiss.txt", "matched_ids.txt", "kp_final.fasta"]:
    if os.path.exists(filename):
        shutil.copy(filename, os.path.join(output_folder, filename))
        print(f"Saved: {filename}")


## 6. Candidate annotation and prioritization

The final non-homologous protein set was inspected using sequence-header annotations. Plasmid-encoded and hypothetical proteins were separated from named chromosomal proteins for downstream manual/literature prioritization.

**Important:** this annotation-based prioritization is a computational screening step, not experimental target validation.


In [ ]:
!grep ">" kp_final.fasta > kp_final_annotations.txt
!wc -l kp_final_annotations.txt
!head -20 kp_final_annotations.txt


In [ ]:
plasmid_count = 0
chromosomal_count = 0
hypothetical_count = 0
named_count = 0

with open("kp_final_annotations.txt") as f:
    for line in f:
        line_lower = line.lower()

        if "(plasmid)" in line_lower:
            plasmid_count += 1
        else:
            chromosomal_count += 1

        if "hypothetical protein" in line_lower:
            hypothetical_count += 1
        else:
            named_count += 1

print(f"Plasmid-encoded: {plasmid_count}")
print(f"Chromosomal: {chromosomal_count}")
print(f"Hypothetical: {hypothetical_count}")
print(f"Named/annotated: {named_count}")


In [ ]:
final_candidates = []

with open("kp_final_annotations.txt") as f:
    for line in f:
        line_stripped = line.strip()
        line_lower = line_stripped.lower()

        is_plasmid = "(plasmid)" in line_lower
        is_hypothetical = "hypothetical protein" in line_lower

        if not is_plasmid and not is_hypothetical:
            final_candidates.append(line_stripped)

print(f"Chromosomal + named candidates: {len(final_candidates)}")

with open("kp_priority_candidates.txt", "w") as out:
    out.write("\n".join(final_candidates))

for line in final_candidates[:30]:
    print(line)


### Functional keyword prioritization

Candidates were grouped using annotation keywords to support manual review. This is a **prioritization heuristic**, not a formal essentiality or druggability prediction.


In [ ]:
tier1_keywords = [
    "adhesin", "fimbri", "hemolysin", "toxin", "secretion", "dsba",
    "hemin", "siderophore", "iron", "porin", "efflux", "permease",
    "transporter", "pts family", "autotransporter", "invasin",
    "capsule", "lipopolysaccharide", "lps"
]

deprioritize_keywords = [
    "ribosomal", "trna", "aminoacyl", "rna polymerase subunit",
    "dna polymerase", "elongation factor", "initiation factor",
    "chaperone", "heat shock"
]

vague_keywords = [
    "putative cytoplasmic protein", "putative inner membrane protein",
    "domain-containing protein", "duf", "family protein"
]

tier1, tier2, tier3_deprioritize, tier4_vague = [], [], [], []

with open("kp_priority_candidates.txt") as f:
    for line in f:
        line_s = line.strip()
        line_l = line_s.lower()

        if any(k in line_l for k in tier1_keywords):
            tier1.append(line_s)
        elif any(k in line_l for k in deprioritize_keywords):
            tier3_deprioritize.append(line_s)
        elif any(k in line_l for k in vague_keywords):
            tier4_vague.append(line_s)
        else:
            tier2.append(line_s)

print(f"Tier 1 (virulence/transport/surface): {len(tier1)}")
print(f"Tier 2 (other named): {len(tier2)}")
print(f"Tier 3 (housekeeping/translation): {len(tier3_deprioritize)}")
print(f"Tier 4 (vague/uncharacterized): {len(tier4_vague)}")

with open("kp_tier1_candidates.txt", "w") as out:
    out.write("\n".join(tier1))


In [ ]:
for filename in [
    "kp_final_annotations.txt",
    "kp_priority_candidates.txt",
    "kp_tier1_candidates.txt"
]:
    if os.path.exists(filename):
        shutil.copy(filename, os.path.join(output_folder, filename))
        print(f"Saved: {filename}")


## 7. Target sequence extraction

Three Tier-1 candidates were shortlisted for closer review: **DsbA** (`YP_005224660.1`), **Irp3** (`YP_005227768.1`), and **ClpV1** (`YP_005227286.1`, a type VI secretion ATPase). Extracting full sequences is a quick way to catch broken annotations before committing further analysis time to a candidate.


In [ ]:
target_ids = ["YP_005224660.1", "YP_005227768.1", "YP_005227286.1"]

def extract_sequences(fasta_file, ids_wanted, output_file):
    write_flag = False

    with open(fasta_file) as infile, open(output_file, "w") as outfile:
        for line in infile:
            if line.startswith(">"):
                seq_id = line[1:].split()[0]
                write_flag = seq_id in ids_wanted

            if write_flag:
                outfile.write(line)

extract_sequences("kp_final.fasta", target_ids, "top3_targets.fasta")

print("Shortlisted candidate sequences:")
!cat top3_targets.fasta


**ClpV1 dropped.** The extracted `YP_005227286.1` sequence is only 72 aa, far short of the ~800–900 aa expected for a genuine ClpV1/type VI secretion ATPase — a truncated or misannotated ORF, not a real full-length candidate. DsbA and Irp3 are the two targets carried forward into structure prediction and screening.


In [ ]:
target_ids = ["YP_005224660.1", "YP_005227768.1"]  # ClpV1 dropped — truncated fragment

extract_sequences("kp_final.fasta", target_ids, "final_2_targets.fasta")

print("Extracted target sequences:")
!cat final_2_targets.fasta


In [ ]:
for filename in ["top3_targets.fasta", "final_2_targets.fasta"]:
    if os.path.exists(filename):
        shutil.copy(filename, os.path.join(output_folder, filename))
        print(f"Saved: {filename}")


## 8. Exploratory structural homolog check (DsbA vs. PDB 4MCU)

A real experimental PDB structure exists for *K. pneumoniae*/*variicola* DsbA (**4MCU**), so before turning to ColabFold, a quick BLAST of the DsbA query against a single reference chain (4MCU, chain A) was used to check whether direct homology modeling from this template was viable.


In [ ]:
# Pull the DsbA sequence out of final_2_targets.fasta
sequences = {}
current_id, current_seq = None, []

with open("final_2_targets.fasta") as infile:
    for line in infile:
        line = line.strip()
        if line.startswith(">"):
            if current_id:
                sequences[current_id] = "".join(current_seq)
            current_id = line[1:].split()[0]
            current_seq = []
        else:
            current_seq.append(line)
    if current_id:
        sequences[current_id] = "".join(current_seq)

with open("dsba_query.fasta", "w") as f:
    f.write(">YP_005224660.1\n" + sequences["YP_005224660.1"] + "\n")

# Reference chain sequence (PDB 4MCU, chain A)
with open("4mcu_chainA.fasta", "w") as f:
    f.write(">4MCU_A\nSNAQITDGKQYITLDKPIAGEPQVLEFFSFYCPHCYQFEEVLHVSDNVRQKLPEGTKMTKYHVEFLGPLGKDLTQAWAVAIALGVEDKITAPMFEAVQKTQTVQSVADIRKVFVDAGVKGEDYDAAWNSFVVKSLVAQQEKAAADLQLQGVPAMYVNGKYQLNPQGMDTSNMDVFVAQYADTVKQLVEKK\n")


In [ ]:
# Build a tiny "database" from the 4MCU sequence, then BLAST the DsbA query against it
!makeblastdb -in 4mcu_chainA.fasta -dbtype prot -out 4mcu_db
!blastp -query dsba_query.fasta -db 4mcu_db -outfmt 6


The alignment is weak and fragmented (one hit has an e-value of ~5.0, consistent with noise rather than a genuine match), so direct template-based homology modeling from 4MCU was judged not viable. Both targets were instead predicted with ColabFold (see README, Section 2).


## 9. Reproducibility note

The following preprocessing occurred **before this Colab notebook**:

1. Retrieval of the RefSeq *K. pneumoniae* protein FASTA (`GCF_000240185.1`).
2. CD-HIT redundancy reduction in WSL.
3. Generation of the non-redundant input `kp_nr.fasta`.

The Colab notebook begins with `kp_nr.fasta` and performs the downstream BLAST-based filtering and target prioritization documented above.
